# GNN — Protein structural-class classification (25PDB)
Τρέξε τα κουτάκια **με τη σειρά**, από πάνω προς τα κάτω. Πάτα το ▶ αριστερά κάθε κουτιού (ή Shift+Enter).

**ΠΡΙΝ ΞΕΚΙΝΗΣΕΙΣ — ενεργοποίησε GPU:**
Πάνω δεξιά μενού → *Runtime* → *Change runtime type* → *Hardware accelerator* → **GPU** → *Save*.

## 1. Έλεγχος GPU + εγκατάσταση torch_geometric

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
!pip -q install torch_geometric
print('torch_geometric installed.')

## 2. Ανέβασε τα δύο αρχεία δεδομένων
Θα ανοίξει κουμπί *Choose Files*. Διάλεξε **PSI_PRED.txt** και **data.txt** (μαζί).

In [ ]:
from google.colab import files
up = files.upload()
print('Uploaded:', list(up.keys()))

## 3. Τρέξε την εκπαίδευση (15 splits)
Με GPU κάθε split είναι λίγα λεπτά. Σώζει μετά από **κάθε** split.

In [ ]:
# ==== GNN training (ASAPooling) — Colab GPU version ====
import copy, gc, time
from datetime import datetime
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, ASAPooling, JumpingKnowledge, global_mean_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

print(f"Start: {datetime.now().strftime('%H:%M:%S')}")

SEQ_FILE="PSI_PRED.txt"; LABELS_TXT="data.txt"
OUT_XLSX="gnn_results.xlsx"; OUT_PNG="conf_gnn_asap.png"

N_SPLITS=15            # Zervou uses 15 for large datasets like 25PDB
EPOCHS_MAX=100; PATIENCE=10   # early stopping exactly like the paper
HIDDEN=64; NUM_LAYERS=3; RATIO=0.90; DROPOUT=0.40
LR=0.001; WEIGHT_DECAY=0.001; BATCH=16; SEED=2
torch.manual_seed(SEED); np.random.seed(SEED)

STATE_CODE={"H":0.0,"C":1.0,"E":2.0}
VERT={"H":(0.0,0.0),"C":(0.5,np.sqrt(3)/2),"E":(1.0,0.0)}
def cgr(seq):
    x,y=0.5,np.sqrt(3)/6; xs,ys=[],[]
    for ch in seq:
        vx,vy=VERT[ch]; x,y=(x+vx)/2,(y+vy)/2; xs.append(x); ys.append(y)
    return np.array(xs),np.array(ys)
def hvg_edges(s):
    n=len(s); E=set()
    for i in range(n):
        cur=float("-inf")
        for j in range(i+1,n):
            if cur<min(s[i],s[j]): E.add((i,j))
            cur=max(cur,s[j])
            if cur>=s[i]: break
    return E
def build_graph(seq,label):
    xs,ys=cgr(seq); E=hvg_edges(xs)&hvg_edges(ys)
    state=np.array([STATE_CODE[c] for c in seq],dtype=np.float32)
    feat=np.column_stack([state,xs,ys]).astype(np.float32)
    x=torch.tensor(feat,dtype=torch.float)
    if E:
        ei=np.array(list(E)).T
        edge_index=torch.tensor(np.hstack([ei,ei[::-1]]),dtype=torch.long)
    else:
        edge_index=torch.empty((2,0),dtype=torch.long)
    return Data(x=x,edge_index=edge_index,y=torch.tensor([label],dtype=torch.long))

print("Loading sequences and labels ...")
with open(SEQ_FILE,encoding="utf-8") as f:
    seqs=[ln.strip() for ln in f if ln.strip()]
lab_raw=pd.read_csv(LABELS_TXT,header=None).iloc[:,-1].astype(str).str.strip()
classes=sorted(lab_raw.unique())
labels=lab_raw.map({c:i for i,c in enumerate(classes)}).to_numpy()
assert len(seqs)==len(labels)
print(f"  {len(seqs)} proteins | classes {classes}")
print("Building graphs ...")
dataset=[build_graph(s,y) for s,y in zip(seqs,labels)]
IN_DIM=dataset[0].x.shape[1]
print(f"  built {len(dataset)} graphs (node feature dim = {IN_DIM})")

class PoolNet(torch.nn.Module):
    def __init__(self,in_dim,hidden,n_classes,num_layers,ratio,p):
        super().__init__()
        self.convs=torch.nn.ModuleList(); self.pools=torch.nn.ModuleList()
        self.convs.append(GCNConv(in_dim,hidden))
        for _ in range(num_layers-1): self.convs.append(GCNConv(hidden,hidden))
        for _ in range(num_layers): self.pools.append(ASAPooling(hidden,ratio=ratio))
        self.jump=JumpingKnowledge(mode="cat")
        self.lin1=Linear(num_layers*hidden,hidden); self.lin2=Linear(hidden,n_classes); self.p=p
    def forward(self,x,edge_index,batch):
        xs=[]
        for conv,pool in zip(self.convs,self.pools):
            x=F.relu(conv(x,edge_index))
            x,edge_index,_,batch,_=pool(x,edge_index,batch=batch)
            xs.append(global_mean_pool(x,batch))
        x=self.jump(xs); x=F.relu(self.lin1(x))
        x=F.dropout(x,p=self.p,training=self.training)
        return self.lin2(x)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  running on: {device}")
crit=torch.nn.CrossEntropyLoss()
def epoch_pass(model,loader,opt=None):
    train=opt is not None; model.train() if train else model.eval()
    tot=correct=total=0
    for data in loader:
        data=data.to(device)
        if train: opt.zero_grad()
        out=model(data.x,data.edge_index,data.batch); loss=crit(out,data.y)
        if train: loss.backward(); opt.step()
        tot+=loss.item()*data.num_graphs
        correct+=(out.argmax(1)==data.y).sum().item(); total+=data.num_graphs
    return tot/total, correct/total
@torch.no_grad()
def collect_preds(model,loader):
    model.eval(); yt,yp=[],[]
    for data in loader:
        data=data.to(device); out=model(data.x,data.edge_index,data.batch)
        yp.extend(out.argmax(1).cpu().tolist()); yt.extend(data.y.cpu().tolist())
    return yt,yp

lbl_idx=list(range(len(classes)))
glabels=["\u03b1","\u03b2","\u03b1/\u03b2","\u03b1+\u03b2"]
def save_results(rows,all_true,all_pred):
    if not rows: return
    per_split=pd.DataFrame(rows); acc=per_split["test_accuracy_%"].to_numpy()
    summary=pd.DataFrame([{"n_splits":len(rows),"mean_accuracy_%":round(acc.mean(),2),
        "std_%":round(acc.std(),2),"worst_%":round(acc.min(),2),"best_%":round(acc.max(),2)}])
    cm_all=confusion_matrix(all_true,all_pred,labels=lbl_idx)
    cm_df=pd.DataFrame(cm_all,index=[f"true_{c}" for c in classes],columns=[f"pred_{c}" for c in classes])
    rep=classification_report(all_true,all_pred,labels=lbl_idx,target_names=classes,
        digits=3,zero_division=0,output_dict=True)
    per_class=pd.DataFrame(rep).transpose().reset_index().rename(columns={"index":"class"})
    with pd.ExcelWriter(OUT_XLSX) as w:
        per_split.to_excel(w,sheet_name="per_split",index=False)
        summary.to_excel(w,sheet_name="summary",index=False)
        cm_df.to_excel(w,sheet_name="confusion_matrix")
        per_class.to_excel(w,sheet_name="per_class",index=False)
    plt.rcParams.update({"font.family":["DejaVu Sans"],"savefig.dpi":300,"savefig.bbox":"tight"})
    C=cm_all.astype(float); row=C.sum(1,keepdims=True); row[row==0]=1; Nrm=C/row
    fig,ax=plt.subplots(figsize=(6.2,5.2)); im=ax.imshow(Nrm,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(glabels); ax.set_yticklabels(glabels)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    ax.set_title("Confusion matrix, mdHVG-AND GNN (ASAPooling)",fontweight="bold",pad=12)
    for i in range(4):
        for j in range(4):
            v=Nrm[i,j]; col="white" if v>0.55 else "#222222"
            ax.text(j,i,f"{int(C[i,j])}\n{v*100:.1f}%",ha="center",va="center",color=col,fontsize=11)
    cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04); cb.set_label("Row-normalized (recall)")
    ax.set_xticks(np.arange(-.5,4,1),minor=True); ax.set_yticks(np.arange(-.5,4,1),minor=True)
    ax.grid(which="minor",color="white",linewidth=1.5); ax.tick_params(which="minor",length=0)
    fig.savefig(OUT_PNG); plt.close(fig)

print(f"Training over {N_SPLITS} random 80/10/10 splits ...")
y_all=labels; rows=[]; all_true=[]; all_pred=[]
for split in range(N_SPLITS):
    _ts=time.time()
    try:
        tr,tmp=train_test_split(range(len(dataset)),test_size=0.20,stratify=y_all,random_state=split)
        va,te=train_test_split(tmp,test_size=0.50,stratify=y_all[tmp],random_state=split)
        mk=lambda idx,sh=False:DataLoader([dataset[i] for i in idx],batch_size=BATCH,shuffle=sh)
        tr_l,va_l,te_l=mk(tr,True),mk(va),mk(te)
        model=PoolNet(IN_DIM,HIDDEN,len(classes),NUM_LAYERS,RATIO,DROPOUT).to(device)
        opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
        best_val,best_state,wait=float("inf"),None,0
        for ep in range(1,EPOCHS_MAX+1):
            epoch_pass(model,tr_l,opt); val_loss,_=epoch_pass(model,va_l)
            if val_loss<best_val-1e-4: best_val,best_state,wait=val_loss,copy.deepcopy(model.state_dict()),0
            else:
                wait+=1
                if wait>=PATIENCE: break
        model.load_state_dict(best_state); _,te_acc=epoch_pass(model,te_l)
        rows.append({"split":split+1,"test_accuracy_%":round(te_acc*100,2),"stopped_epoch":ep})
        print(f"  split {split+1:2d}/{N_SPLITS}: test acc {te_acc*100:5.2f}%  (epoch {ep}, {time.time()-_ts:.0f}s)")
        yt,yp=collect_preds(model,te_l); all_true.extend(yt); all_pred.extend(yp)
        save_results(rows,all_true,all_pred)
        print(f"    [saved progress: {len(rows)} split(s)]")
    except RuntimeError as e:
        print(f"  !! split {split+1} failed: {e}")
    finally:
        for nm in ("model","opt","tr_l","va_l","te_l","best_state"):
            if nm in dir():
                try: del globals()[nm]
                except Exception: pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if rows:
    acc=np.array([r["test_accuracy_%"] for r in rows],dtype=float)
    print(f"\nMean over {len(rows)} splits: {acc.mean():.2f}% +/- {acc.std():.2f}")
    print(f"Saved -> {OUT_XLSX} and {OUT_PNG}")
print(f"End: {datetime.now().strftime('%H:%M:%S')}")


## 4. Κατέβασε τα αποτελέσματα

In [ ]:
from google.colab import files
files.download('gnn_results.xlsx')
files.download('conf_gnn_asap.png')